# Telugu forced alignment on Colab (CPU, High-RAM)

Runtime → Change runtime type → **CPU** + **High-RAM** (no GPU needed). Runs the same `mfa train` step as `project/Telugu/preprocess.py --stage align` (MFA 3.4.3, trained on this corpus with the rule-based dictionary from `fs2ra/text_te.py`).

Input (upload once to `MyDrive/fs2ra/`): `telugu_corpus.tar` — made locally after `preprocess.py --stage export` with
`cd data && tar -cf ../dist/telugu_corpus.tar telugu/corpus telugu/dictionary.txt`.

Output to `MyDrive/fs2ra/`: `telugu_alignment.tar` (`telugu/TextGrid_mfa/` + the trained `mfa_telugu_acoustic.zip`). Extract it into the local `data/` folder, then run `preprocess.py --stage features` locally.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fs2ra"
if not os.path.exists("/content/data/telugu/dictionary.txt"):
    os.makedirs("/content/data", exist_ok=True)
    !cp $DRIVE/telugu_corpus.tar /content/ && tar -xf /content/telugu_corpus.tar -C /content/data && rm /content/telugu_corpus.tar
!ls /content/data/telugu/corpus/te_male | wc -l; nproc; free -g | head -2

In [ ]:
# MFA needs Kaldi binaries from conda-forge: self-contained micromamba env (~5-10 min)
if not os.path.exists("/content/mfa/bin/mfa"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/mfa -c conda-forge montreal-forced-aligner=3.4.3
!/content/bin/micromamba run -p /content/mfa mfa version

In [ ]:
# Same command as fs2ra/pipeline.py::stage_align with align.train_acoustic_model.
# Runs in the background (nohup): a browser/wifi disconnect does not stop it, and the
# result tar is written to Drive as soon as alignment finishes. Progress: next cell.
D = "/content/data/telugu"
JOBS = os.cpu_count()
LOG = "/content/mfa_train.log"
if os.system("pgrep -f 'mfa train' > /dev/null") == 0:
    print("mfa train is already running - just use the next cell to watch it")
else:
    !nohup bash -c "/content/bin/micromamba run -p /content/mfa mfa train $D/corpus $D/dictionary.txt $D/mfa_telugu_acoustic.zip --output_directory $D/TextGrid_mfa --clean --num_jobs $JOBS --output_format long_textgrid && cd /content/data && tar -cf $DRIVE/telugu_alignment.tar telugu/TextGrid_mfa telugu/mfa_telugu_acoustic.zip && echo ALL_DONE" > $LOG 2>&1 &
    print("started; log:", LOG)


In [ ]:
# Re-run this cell any time (e.g. after reconnecting) to see progress.
# Finished when the last line is ALL_DONE; then telugu_alignment.tar is in MyDrive/fs2ra/.
!tail -c 1500 /content/mfa_train.log; echo; pgrep -f 'mfa train' > /dev/null && echo "== still running ==" || echo "== not running =="
!find /content/data/telugu/TextGrid_mfa -name '*.TextGrid' 2>/dev/null | wc -l
!ls -la $DRIVE/telugu_alignment.tar 2>/dev/null
